# ND-AWS: warm-started QAOA inside the NDAR loop

[C02](../C_gauge_symmetries_and_NDAR/C02_run_qaoa_optimization_with_NDAR.ipynb) ran NDAR with a
plain QAOA sampler: after every round it re-gauges the Hamiltonian so that the best solution
found so far is relabelled to |00...0>, the state biased readout noise pulls towards. This
notebook changes the local sampler to a **warm-started** p=1 QAOA circuit submitted to a Qiskit
simulator. That combination is **ND-AWS -- Noise-Directed Adaptive Warm-Starting**.

The two mechanisms point the same way. The warm start biases the initial product state towards
|00...0>, and NDAR keeps moving the current best solution onto |00...0>, so each iteration
starts the circuit near the best solution known and lets the noise pull towards it too.
[D01](D01_warm_start_distance_to_solution.ipynb) measures what that bias is worth and what it
costs when it points at the wrong bitstring; ND-AWS is the loop that moves it towards the right
one.

The run below is noisy. It is then compared with plain, noiseless p=1 QAOA given the same
total budget, which is the comparison
[C02](../C_gauge_symmetries_and_NDAR/C02_run_qaoa_optimization_with_NDAR.ipynb) makes for NDAR.
Both sides set their angles offline on the same simulator, so what the comparison measures is
the ansatz and the loop, not the angle-setting method.

**What to change**: the instance in the first configuration cell; `probability_1_to_0` (the
readout bias NDAR exploits); `number_of_samples_qiskit` and
`number_of_function_calls_simulator` (the cost of one NDAR iteration);
`max_unsuccesfull_iterations` and `bias_list_iteration_callable` (the loop itself).

**Writes**: the Hamiltonian instance and every NDAR iteration's results into the standardized
result store (`DEFAULT_STORAGE_DIRECTORY`; with that variable unset it is `output/` at the
repository root), and the performance plot as an HTML file into `../temp/`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
#Qiskit reports every gate-duration rounding as a user warning, which floods the output.
import warnings

import numpy as np
from dotenv import load_dotenv
from qiskit_aer.noise import NoiseModel, ReadoutError

from quapopt import ancillary_functions as anf
from quapopt.ancillary_functions.presets import generate_random_hamiltonian
from quapopt.ancillary_functions.presets.ndar import (build_local_sampler_callable_and_updater_NDAR,
                                                      build_universal_logging_callable,
                                                      LocalSamplerName)
from quapopt.ancillary_functions.presets.wrapped_optimization import get_classical_optimizer_p1_default
from quapopt.circuits import backend_utilities as bck_utils
from quapopt.circuits.backend_utilities.qiskit.qiskit_config import DEFAULT_SIMULATOR_BACKEND_KWARGS
from quapopt.circuits.gates.logical.LogicalGateBuilderQiskit import LogicalGateBuilderQiskit
from quapopt.data_analysis.data_handling import (HamiltonianModels,
                                                 ResultsLogger,
                                                 STANDARD_NAMES_DATA_TYPES as SNDT,
                                                 STANDARD_NAMES_VARIABLES as SNV)
from quapopt.meta_algorithms.NDAR import ConvergenceCriterion, ConvergenceCriterionNames
from quapopt.meta_algorithms.NDAR.NDARRunner import NDARRunner
from quapopt.optimization.QAOA import (InitialStateType,
                                       MixerType,
                                       PhaseSeparatorType,
                                       QAOAFunctionInputFormat,
                                       QubitMappingType)
from quapopt.optimization.QAOA.implementation.QAOARunnerSampler import QAOARunnerSampler
from quapopt.optimization.QAOA.simulation.QAOARunnerExpValuesRDMs import QAOARunnerExpValuesRDMs
from quapopt.optimization.parameter_setting.variational.QAOAOptimizationRunner import QAOAOptimizationRunner

warnings.filterwarnings("ignore", category=UserWarning, module='qiskit')

os.makedirs('../temp', exist_ok=True)

### Generate a random Hamiltonian instance

* One instance for the whole notebook, from the `generate_random_hamiltonian` shortcut
  [A01](../A_generating_hamiltonians/A01_generate_and_save_Hamiltonians.ipynb) introduces.
* The cells below report approximation ratios, which need both extremal energies, so this
  notebook solves the instance. `write_to_file()` caches the solve in the store, so a second
  run of this notebook does not repeat it.
* **What to change here**: `number_of_qubits`, `seed_cost_hamiltonian` and `edge_probability`.
  At 25 qubits or fewer the solve is exact; above that it falls back to a heuristic, and the
  extremal energies become a bound rather than the optimum.
* **Result**: `cost_hamiltonian`, `ground_state_energy` and `highest_energy`.


In [ ]:
hamiltonian_model = HamiltonianModels.ErdosRenyi
edge_probability = 0.5
localities = (2,)

#The circuits run on a statevector simulator, whose memory doubles with every qubit.
number_of_qubits = 24
#Without a GPU, the exact solve and the statevector simulation run on the CPU, and their memory
#and run time double with every added qubit.
from quapopt import AVAILABLE_SIMULATORS
if number_of_qubits > 20 and not {'cupy', 'cuda', 'torch'} & set(AVAILABLE_SIMULATORS):
    print(f"WARNING: no GPU found, and number_of_qubits = {number_of_qubits}. On a CPU, memory and run time "
          f"double with every added qubit; at 24 qubits this notebook needs more than 10 GiB of memory. "
          f"For a CPU run, set number_of_qubits = 20 in this cell (a few minutes).")
seed_cost_hamiltonian = 0

cost_hamiltonian = generate_random_hamiltonian(hamiltonian_model=hamiltonian_model,
                                               number_of_qubits=number_of_qubits,
                                               seed=seed_cost_hamiltonian,
                                               default_backend='numpy',
                                               localities=localities,
                                               instance_kwargs={'p_or_M': edge_probability})

if cost_hamiltonian.ground_state_energy is None or cost_hamiltonian.highest_energy is None:
    cost_hamiltonian.solve_hamiltonian(both_directions=True)
#Keeps the solve with the instance, so a re-run reads it instead of solving again.
cost_hamiltonian.write_to_file()

ground_state_energy = float(cost_hamiltonian.ground_state_energy)
highest_energy = float(cost_hamiltonian.highest_energy)

print("Class description:", cost_hamiltonian.hamiltonian_class_description)
print("Instance description:", cost_hamiltonian.hamiltonian_instance_description)
print("Ground state energy:", ground_state_energy, " highest energy:", highest_energy)

### The backend and the readout attractor

* Everything here runs on the local `aer` simulator, so no quantum-hardware account is needed.
  `load_dotenv()` reads a local `.env` if there is one, and the check below reports what it
  found instead of letting a provider constructor raise deep inside Qiskit.
* **NDAR needs a noise attractor**: a bias in the sampling that pulls measured bitstrings
  towards one particular state. The model below loses an excitation ten times more often than
  it creates one, so the samples drift towards |00...0> -- the state NDAR relabels the best
  solution to.
* **What to change here**: `probability_1_to_0` and `probability_0_to_1`, to strengthen or
  weaken the attractor.
* **Result**: `qiskit_backend` and `qiskit_sampler_options`.


In [ ]:
backend_name = 'aer'
simulation = True
#For larger system sizes, use mps simulation
mps_simulation = False
max_bond_dimension = 40

load_dotenv()
HAVE_IBM = any(os.getenv(k) for k in ('IBM_TOKEN', 'IBM_CREDENTIALS_PATH', 'IBM_ACCOUNT_NAME'))
if HAVE_IBM:
    print("IBM credentials found, but this notebook is a local simulation: using 'aer'")
else:
    print("IBM credentials absent: using 'aer'")

qiskit_backend = bck_utils.get_qiskit_backend(backend_name=backend_name, qiskit_provider=None)

#The attractor: losing an excitation is ten times likelier than gaining one.
probability_1_to_0 = 0.10
probability_0_to_1 = 0.01
readout_noise_model = NoiseModel()
readout_noise_model.add_all_qubit_readout_error(
    ReadoutError([[1 - probability_0_to_1, probability_0_to_1],
                  [probability_1_to_0, 1 - probability_1_to_0]]))

seed_main = 1
qiskit_sampler_options = DEFAULT_SIMULATOR_BACKEND_KWARGS.copy()
qiskit_sampler_options['noise_model'] = readout_noise_model
qiskit_sampler_options['seed_simulator'] = seed_main

print('backend:', qiskit_backend.name)

### The warm-started ansatz, its angles and its bias schedule

* The ansatz is a p=1 warm-started QAOA circuit: a standard phase separator, and a mixer and
  initial state of the warm-started family. `ws_qaoa_identical` is what makes the warm start
  follow the gauges NDAR proposes, which is the ND-AWS part.
* Angles are set offline. For each iteration the sampler optimizes gamma on a
  reduced-density-matrix simulator with COBYQA plus basinhopping, and finds the best beta
  internally by a scan over `betas_search_space_size` values, so only gamma reaches the
  optimizer. Those calls cost simulator time, not shots.
* The bias `c` is the per-qubit weight of the warm-started initial product state: each qubit is
  prepared with probability `c` of being measured against its warm-start value, through a
  rotation by `2 arcsin(sqrt(c))`. So `c = 0.5` is the unbiased |+> state and a smaller `c` sits
  closer to the warm start. **A list means both values are tried and the better one kept**, so a
  two-value entry costs twice `number_of_samples_qiskit` shots in that iteration.
* **What to change here**: `number_of_function_calls_simulator` and `betas_search_space_size`
  buy angle quality with simulator time, `number_of_samples_qiskit` is the shots per angle
  setting, and `bias_list_iteration_callable` is the trajectory from the unbiased state towards
  the warm start.
* **Result**: the ansatz settings and `classical_optimizer`. Nothing runs yet.


In [ ]:
ansatz_depth = 1
phase_separator_type = PhaseSeparatorType.QAOA
initial_state_type = InitialStateType.ws_qaoa_identical
mixer_type = MixerType.ws_qaoa_identical

#The share of the largest-magnitude interactions one p=1 layer implements. With all-to-all
#connectivity there is no reason not to implement the whole Hamiltonian, so the subset seed
#never reaches a circuit.
time_block_size = 1.0
time_block_seed = -1

#Shots per angle setting, and the offline angle-search budget per iteration. The shot count is
#deliberately small: with a few thousand shots per iteration the loop reaches this instance's
#optimum in its first iterations and the curve is flat, which shows nothing.
number_of_samples_qiskit = 100
number_of_function_calls_simulator = 100
betas_search_space_size = 10 ** 5
efficient_finding_of_best_beta = True

#Only gamma reaches this optimizer; beta comes from the internal scan.
classical_optimizer, optimizer_metadata = get_classical_optimizer_p1_default(
    efficient_finding_of_best_beta=efficient_finding_of_best_beta,
    number_of_qubits=number_of_qubits)


def bias_list_iteration_callable(ndar_iteration_index):
    """The warm-start bias per iteration; a list tries both values and keeps the better one."""
    if ndar_iteration_index == 0:
        return [0.5]
    elif ndar_iteration_index <= 3:
        return [0.1, 0.05]
    return [0.05, 0.025]

### The NDAR loop's own settings

* `MaxUnsuccessfulTrials = N` stops the loop after `N` iterations in a row that fail to improve
  on the best energy so far, so a run that keeps improving keeps going.
* `break_after_finding_ground_state` is on in the run cell below, so a run that reaches the
  known optimum stops there instead of spending the rest of its budget.
* **What to change here**: `max_unsuccesfull_iterations`, and `experiment_folders_hierarchy` to
  put this run's rows somewhere else in the store.
* **Result**: `convergence_criterion`, `logger_kwargs_ndar` and the experiment set id every row
  below is stored under.


In [ ]:
max_unsuccesfull_iterations = 3

convergence_criterion = ConvergenceCriterion(
    convergence_criterion_name=ConvergenceCriterionNames.MaxUnsuccessfulTrials,
    convergence_value=max_unsuccesfull_iterations)

experiment_set_id = f"{anf.create_random_uuid()}"
logger_kwargs_ndar = {'experiment_folders_hierarchy': ['SimulationResults', 'TestingNDAWS'],
                      'experiment_set_id': experiment_set_id,
                      }

print("Experiment set id:", experiment_set_id)

### The local sampler, and the ND-AWS run

* NDAR treats its inner optimizer as a black box: a "local sampler" that takes a list of
  Hamiltonian representations and returns the best solutions it found.
  `build_local_sampler_callable_and_updater_NDAR` packages the warm-started Qiskit sampler as
  one, so the call below is the same `NDARRunner` call
  [C02](../C_gauge_symmetries_and_NDAR/C02_run_qaoa_optimization_with_NDAR.ipynb) makes, with a
  different sampler inside it.
* The three dictionaries are that sampler's arguments: `ansatz_config` is the circuit,
  `simulator_config` the offline angle search, `qiskit_config` the compilation and the
  submission.
* A "session" groups the submitted circuits. On a simulator it is mocked, so nothing is sent
  anywhere; on hardware it would be a real IBM Runtime batch.
* **What to change here**: nothing.
* **Result**: `best_result_ndar` and `optimization_history_ndar`, plus one NDAR-overview row
  and every shot of every iteration in the result store.


In [ ]:
#All-to-all connectivity and logical gates: there is no routing to do on a simulator.
seed_transpiler = 42
qiskit_config = {'qubit_mapping_type': QubitMappingType.fully_connected,
                 'qiskit_backend': qiskit_backend,
                 'backend_name': backend_name,
                 'pass_manager_kwargs': {'optimization_level': 0},
                 'pass_manager_seeds_list': [seed_transpiler],
                 'gate_builder': LogicalGateBuilderQiskit(),
                 'qubit_indices_physical': None,
                 'simulation': simulation,
                 'mps_simulation': mps_simulation,
                 'max_bond_dimension': max_bond_dimension,
                 'noiseless_simulation': False,
                 'qiskit_sampler_options': qiskit_sampler_options,
                 }

ansatz_config = {'time_block_size': time_block_size,
                 'time_block_seed': time_block_seed,
                 'phase_separator_type': phase_separator_type,
                 'mixer_type': mixer_type,
                 'initial_state_type': initial_state_type,
                 }

simulator_config = {'classical_optimizer': classical_optimizer,
                    'betas_search_space_size': betas_search_space_size,
                    'efficient_finding_of_best_beta': efficient_finding_of_best_beta,
                    }

with bck_utils.create_qiskit_session(qiskit_backend=qiskit_backend,
                                     mocked=simulation,
                                     mode='batch') as session_ibm:
    qiskit_config['session_ibm'] = session_ibm

    sampler_qiskit, updater_qiskit = build_local_sampler_callable_and_updater_NDAR(
        local_sampler_name=LocalSamplerName.QiskitWithOfflineAnglesP1,
        r_best_results=1,
        number_of_samples_qiskit=number_of_samples_qiskit,
        number_of_function_calls_simulator=number_of_function_calls_simulator,
        optimize_bias_offline=False,
        bias_list_iteration_callable=bias_list_iteration_callable,
        ansatz_config=ansatz_config,
        simulator_config=simulator_config,
        qiskit_config=qiskit_config,
        verbosity=0,
        logger_kwargs_main=logger_kwargs_ndar)

    ndar_runner = NDARRunner(input_hamiltonian_representation=cost_hamiltonian.copy(),
                             convergence_criterion=convergence_criterion,
                             numpy_rng_boltzmann=np.random.default_rng(seed=seed_main))

    best_result_ndar, optimization_history_ndar = ndar_runner.run_NDAR(
        local_sampler_functions=[(sampler_qiskit,
                                  updater_qiskit,
                                  build_universal_logging_callable(),
                                  LocalSamplerName.QiskitWithOfflineAnglesP1.value)],
        optimize_over_r_gauges=1,
        show_progress_bar_ndar=True,
        verbosity=1,
        break_after_finding_ground_state=True)

best_energy_ndaws = float(best_result_ndar[0])
anf.cool_print("BEST ENERGY OF ND-AWS:", f"{best_energy_ndaws} "
                                         f"(ground state {ground_state_energy})", 'green')

### Plain QAOA with the same budget

* Can noisy ND-AWS match plain, noiseless p=1 QAOA given the same budget? The comparison is
  only fair if both sides set their angles the same way. The ND-AWS sampler sets its angles
  **offline**, exactly, on the reduced-density-matrix simulator, and spends its shots on
  sampling alone -- so plain QAOA does the same: the same optimizer, the same simulator, the
  same call budget, and then every one of its shots at the single best angle pair.
* Plain means both ND-AWS ingredients dropped at once: the standard |+...+> initial state
  instead of the warm start, and no re-gauging between rounds. The warm-start bias `c = 0.5`
  IS that standard state, so the same one-parameter optimizer serves here.
* The shot budget is every shot the loop above spent, summed over its bias schedule, and it is
  taken noiselessly. That is the arm ND-AWS has to beat while running with the readout noise
  switched on.
* **What to change here**: nothing.
* **Result**: `best_energy_plain_qaoa`. The cell prints the best energy of both runs.


In [ ]:
#Angles first, offline, the way the ND-AWS sampler sets its own. The bias stays at its default
#0.5, which is |+...+> with the standard X mixer, so this is plain QAOA on the same simulator.
qaoa_runner_plain = QAOARunnerExpValuesRDMs(hamiltonian_representations_cost=[cost_hamiltonian.copy()],
                                            store_full_information_in_history=True,
                                            #Keeps the result objects, so the angles below are reachable by name.
                                            simplified_data_storage=False)
qaoa_optimizer_plain = QAOAOptimizationRunner(qaoa_runner=qaoa_runner_plain)

best_result_plain_qaoa, _ = qaoa_optimizer_plain.run_optimization(
    qaoa_depth=ansatz_depth,
    number_of_function_calls=number_of_function_calls_simulator,
    #Exact expected values: the offline search costs simulator time, not shots.
    number_of_samples=np.inf,
    classical_optimizer=classical_optimizer,
    find_best_beta_only=efficient_finding_of_best_beta,
    betas_search_space_size=betas_search_space_size,
    show_progress_bar=True,
    verbosity=0)

expected_energy_plain_qaoa, (_, qaoa_result_plain) = best_result_plain_qaoa[0]
best_angles_plain_qaoa = qaoa_result_plain.angles

#Every shot the loop above spent, in one noiseless batch at those angles. An iteration that
#tries two bias values pays for both, so the count is summed over the schedule.
number_of_samples_plain_qaoa = number_of_samples_qiskit * sum(
    len(bias_list_iteration_callable(i)) for i in range(len(optimization_history_ndar)))

qaoa_runner_sampler = QAOARunnerSampler(hamiltonian_representations_cost=[cost_hamiltonian.copy()],
                                        hamiltonian_representations_phase=None,
                                        numpy_rng_sampling=np.random.default_rng(seed=seed_main))
qaoa_runner_sampler.initialize_backend_python(backend='auto')

result_plain_qaoa = qaoa_runner_sampler.run_qaoa(best_angles_plain_qaoa,
                                                 input_format=QAOAFunctionInputFormat.direct_vector,
                                                 qaoa_depth=ansatz_depth,
                                                 number_of_samples=number_of_samples_plain_qaoa,
                                                 measurement_noise=None)

best_energy_plain_qaoa = float(result_plain_qaoa.energy_best)

print(f"plain QAOA angles (gamma, beta): {np.round(best_angles_plain_qaoa, 6)}, "
      f"expected energy {expected_energy_plain_qaoa:.6f}, "
      f"{number_of_samples_plain_qaoa} shots taken")
anf.cool_print("BEST ENERGY OF PLAIN QAOA (noiseless):", best_energy_plain_qaoa, 'blue')
anf.cool_print("BEST ENERGY OF ND-AWS (noisy):", best_energy_ndaws, 'red')

### Reading the run back

* The rows could be taken straight from the optimization output, but reading them from the
  store is what a later analysis would do, so that is what happens here: the same
  `ResultsLogger` the loop wrote through, asked for the NDAR overview.
* The second read is every shot the loop recorded. Each shot is stored in the gauge that was
  active when it was taken -- the frame the readout attractor acts on -- so the mean Hamming
  weight of those shots says WHERE the circuit samples. Expect a value near 0: NDAR has
  relabelled the best solution to |00...0>, the warm start biases the circuit towards it, and
  the attractor pulls the same way. A run without the re-gauging would sit near 0.5.
* **What to change here**: nothing.
* **Result**: `ndar_data`, one row per NDAR iteration.


In [ ]:
results_logger = ResultsLogger(**logger_kwargs_ndar)

ndar_data = results_logger.gather_results(data_type=SNDT.NDAROverview)
ndar_data = ndar_data.sort_values(by=SNV.NDARIteration.id_long).reset_index(drop=True)

shots = results_logger.gather_results(data_type=SNDT.Bitstrings)
shots_array = np.array([list(eval(b)) if isinstance(b, str) else list(b)
                        for b in shots[SNV.Bitstring.id_long]], dtype=int)
print(f"{shots_array.shape[0]} shots recorded; mean Hamming weight {shots_array.mean():.4f} "
      f"(0 is the attractor |00...0>, 0.5 is a typical random bitstring)")

ndar_data

### Visualization

* The best energy of each NDAR iteration, as an approximation ratio against this instance's
  extremal energies. The dashed line is the noiseless plain QAOA run of the same budget.
* The curve is non-decreasing by construction: NDAR keeps its best solution.
* **What to change here**: the `update_yaxes` range.
* **Result**: the figure, shown here and written to
  `../temp/NDAWS_performance_qiskit_sampler.html`.


In [ ]:
import plotly
import plotly.graph_objects as go

plotly.io.templates.default = "plotly"
plotly.offline.init_notebook_mode(connected=True)

delta_energy = highest_energy - ground_state_energy
xs = ndar_data[SNV.NDARIteration.id_long].tolist()
ys = [(highest_energy - Ei) / delta_energy
      for Ei in ndar_data[SNV.EnergyBest.id_long].tolist()]
ar_plain_qaoa = (highest_energy - best_energy_plain_qaoa) / delta_energy

ndar_figure = go.Figure()
ndar_figure.add_trace(go.Scatter(x=xs, y=ys, mode='lines+markers',
                                 name=f'ND-AWS, p(0|1) = {probability_1_to_0}'))
#The label sits below the line, where the ND-AWS curve does not run.
ndar_figure.add_hline(y=ar_plain_qaoa, line_dash='dash', line_color='black',
                      annotation_text='plain QAOA, noiseless, same budget',
                      annotation_position='bottom right')

ndar_figure.update_layout(
    title=f"ND-AWS performance for {number_of_qubits} qubits on "
          f"{cost_hamiltonian.hamiltonian_class_description}",
    xaxis_title="NDAR Iteration",
    yaxis_title="Approximation Ratio",
    template="plotly",
    font=dict(size=14),
    hovermode="x unified")
ndar_figure.update_yaxes(range=[min([ar_plain_qaoa, max(ys)])*0.9, 1.03])
ndar_figure.update_xaxes(tickvals=xs)

#The file name names the local sampler, so the C02 figure is not overwritten.
plotly.offline.plot(ndar_figure,
                    filename='../temp/NDAWS_performance_qiskit_sampler.html',
                    auto_open=True)
# ndar_figure.show()